In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import * 
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')

Using cpu


In [6]:
import requests
from io import BytesIO
import certifi

In [7]:
# 전처리
path = 'https://bit.ly/ds_boston_csv'
response = requests.get(path,verify=certifi.where(),timeout=15)
response.raise_for_status()
data = pd.read_csv(BytesIO(response.content))

# x, y 분할
target = 'medv'
features = ['lstat', 'ptratio', 'crim']
x = data.loc[:, features]
y= data.loc[:, target]

# xtrain, val 분할
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=.2, random_state=20)

# 스케일러 선언
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)

In [8]:
# 텐서 변환 + Data Loader 생성 함수
def make_DataSet(x_train, x_val, y_train, y_val, batch_size= 32) :
    x_train_tensor = torch.tensor(x_train, dtype= torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype= torch.float32).view(-1,1)
    x_val_tensor = torch.tensor(x_val, dtype= torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype= torch.float32).view(-1,1)
    
    # TensorDataSet 생성
    train_dataset = TensorDataset(x_train_tensor, y_train_tensor)

    # DataLoader 생성
    train_loader = DataLoader(train_dataset, batch_size= batch_size, shuffle=True)
    return train_loader , x_val_tensor, y_val_tensor

In [9]:
# 넘파이 배열변환 - cpu 연산
y_train = y_train.values
y_val = y_val.values

In [13]:
# 데이터 로더 구성 by make_Dataset
train_loader, x_val_ts, y_val_ts = make_DataSet(x_train, x_val, y_train, y_val, 32)

In [10]:
# 모델 구조 설계
n_feature = x.shape[1]
model1= nn.Sequential(nn.Linear(n_feature, 1)).to(device)
loss_fn = nn.MSELoss()
optimizer = Adam(model1.parameters(), lr=0.01)
print(model1)

Sequential(
  (0): Linear(in_features=3, out_features=1, bias=True)
)


In [15]:
# 학습 함수 정의
def train(dataloader, model, loss_fn, optimizer, device):
    size= len(dataloader.dataset)
    num_batches = len(dataloader)
    tr_loss = 0
    model.train()
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = loss_fn(pred, y)
        tr_loss += loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    tr_loss /= num_batches
    return tr_loss.item()

In [12]:
# 검증 함수 정의
def evaluate(x_val_tensor, y_val_tensor, model, loss_fn, device):
    model.eval()
    with torch.no_grad():
        x, y = x_val_tensor.to(device), y_val_tensor.to(device)
        pred = model(x)
        val_loss = loss_fn(pred, y).item()
    return val_loss, pred

In [18]:
# 주기 50으로 학습
epochs = 50
tr_loss_list, vals_loss_list = [], []
for t in range(epochs):
    tr_loss = train(train_loader, model1, loss_fn, optimizer, device)
    val_loss, _ = evaluate(x_val_ts, y_val_ts, model1, loss_fn, device)
    tr_loss_list.append(tr_loss)
    vals_loss_list.append(val_loss)
    print(f'Epoch {t+1}, train loss {tr_loss:4f}, val loss {val_loss:4f}')

Epoch 1, train loss 581.187744, val loss 503.338043
Epoch 2, train loss 580.966919, val loss 493.338348
Epoch 3, train loss 558.608459, val loss 483.410889
Epoch 4, train loss 553.315796, val loss 473.768311
Epoch 5, train loss 538.906494, val loss 464.278809
Epoch 6, train loss 536.100098, val loss 454.993347
Epoch 7, train loss 525.009399, val loss 445.957611
Epoch 8, train loss 517.681213, val loss 437.047882
Epoch 9, train loss 505.762756, val loss 428.296967
Epoch 10, train loss 497.714233, val loss 419.759155
Epoch 11, train loss 486.754547, val loss 411.345093
Epoch 12, train loss 479.645355, val loss 403.167053
Epoch 13, train loss 480.504547, val loss 395.208801
Epoch 14, train loss 465.674469, val loss 387.336700
Epoch 15, train loss 453.739380, val loss 379.592896
Epoch 16, train loss 445.480194, val loss 372.149170
Epoch 17, train loss 438.998657, val loss 364.849731
Epoch 18, train loss 430.683807, val loss 357.725464
Epoch 19, train loss 424.875214, val loss 350.652039
Ep